<a href="https://colab.research.google.com/github/shanikairoshi/DeepUnfolding-based-FL/blob/main/main_inital_DUQFL_Scaffold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# %%capture
!pip install genomic-benchmarks
!pip install qiskit qiskit_machine_learning qiskit_algorithms qiskit-aer




In [2]:
import sys
from pathlib import Path
PROJ = Path.cwd() / "InitialDQFL_Project"
if str(PROJ) not in sys.path:
    sys.path.insert(0, str(PROJ))
import sys
sys.path.append('/content/drive/MyDrive/InitialDQFL_Project')
# ─── 5. Assemble filenames for each artifact ─────────────────────────────────
drive_root = "/content/drive/MyDrive/InitialDQFL_Project"

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import logging

logging.getLogger("qiskit_machine_learning").setLevel(logging.ERROR)
logging.getLogger("qiskit_machine_learning.neural_networks.sampler_qnn").setLevel(logging.ERROR)

Load and Split data

run federated loop and plot

In [ ]:
import os
import json
from datetime import datetime
from common.imports import *
from configs.dataset_genome import *     # swap to other configs as needed
#from io_utils.naming import stamp_now, flags, build_param_str, make_filenames
from configs.base_config import *
#from io_utils.naming import build_param_str, stamp_now, make_filenames
from training.data_factory import *
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from training.scaffold_utils import *
from training.metrics import *
from training.fedprox_utils import *

from ml.meta_controller import MetaConfig



# ---------------- method selection ----------------
# Uncomment the desired method name
method_name = "duqfl"   # "duqfl" | "fedprox" | "scaffold" | "fedavg_fixed_spsa" | "fedavg_tuned_adam"

fedprox_mu = 1e-2
scaffold_lr = 0.1
local_maxiter = 25

# ---------------- build data ----------------
clients, test_sequences, test_labels, num_features = build_clients_and_meta(
    dataset_name=dataset_name,
    split_type=split_type,
    num_clients=num_clients,
    num_epochs=num_epochs,
    samples_per_epoch=samples_per_epoch,
    word_size=word_size,
    global_seed=global_seed,
    mnist_n_features=mnist_n_features,
    mnist_digit_a=mnist_digit_a,
    mnist_digit_b=mnist_digit_b,
    breast_pca_n_features=breast_pca_n_features,
    non_iid_ratio=non_iid_ratio,
    quantity_variation=quantity_variation,
    noniid_seed=noniid_seed,
)

X_val, y_val = test_sequences, test_labels

# ---------------- run folder ----------------
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
run_name = f"{dataset_name}_{split_type}_{stamp}"
method_root = os.path.join(drive_root, "results", method_name, run_name)
os.makedirs(method_root, exist_ok=True)

# ---------------- method state ----------------
scaffold_state = None
param_dim = len(RealAmplitudes(num_qubits=len(test_sequences[0]), reps=3).parameters)

if method_name == "scaffold":
    scaffold_state = init_scaffold_state(
        num_clients=len(clients),
        param_dim=param_dim,
    )

meta_cfg = MetaConfig(
    outer_lr=outer_lr,
    outer_radius=outer_radius,
    update_every=meta_update_every,
    lambda_fair=lambda_fair,
    lambda_comm=lambda_comm,
    lambda_stab=lambda_stab,
    lambda_compat=lambda_compat,
    subset_clusters=subset_clusters,
    subset_top_k=subset_top_k,
    subset_tau=subset_tau,
    subset_min_clients=subset_min_clients,
)

# ============================
# TEMPORARY DUQFL DEBUG CONFIG
# ============================
extra_config = {
    "method_name": "duqfl",
    "local_maxiter": 25,
    "duqfl_maxiter_per_unfold": 5,
}

# Disable trajectory subset selection
meta_cfg.subset_top_k = None
meta_cfg.subset_top_k = None
meta_cfg.subset_tau = 0.0
meta_cfg.update_every = 1
meta_cfg.outer_lr = 1e-2
meta_cfg.outer_radius = 2e-2

use_outer_meta = True
'''
extra_config = {
    "method_name": method_name,
    "fedprox_mu": fedprox_mu if method_name == "fedprox" else None,
    "scaffold_state": scaffold_state if method_name == "scaffold" else None,
    "scaffold_lr": scaffold_lr if method_name == "scaffold" else None,
    "local_maxiter": local_maxiter,
    "duqfl_maxiter_per_unfold": 5,
}
'''
# TEMPORARY DEBUGGING
meta_cfg.subset_top_k = None
meta_cfg.subset_tau = 0.0

# Workaround: Make extra_config available in builtins for training.loop to access during import
import builtins
builtins.extra_config = extra_config

# Now import training.loop after extra_config and its dependencies are defined
from training.loop import *

# ---------------- output files ----------------
global_csv_file = os.path.join(method_root, "global_accuracies.csv")
validation_csv_file = os.path.join(method_root, "validation.csv")
round_metrics_csv_file = os.path.join(method_root, "round_metrics.csv")
client_trace_csv_file = os.path.join(method_root, "client_unfold_trace.csv")
subset_csv_file = None if method_name != "duqfl" else os.path.join(method_root, "subset_selection.csv")
subset_centroid_csv_file = None if method_name != "duqfl" else os.path.join(method_root, "subset_centroid.csv")
outer_meta_csv_file = None if method_name != "duqfl" else os.path.join(method_root, "outer_meta.csv")
global_params_npz_file = os.path.join(method_root, "global_params.npz")

# ---------------- config record ----------------
run_config = {
    "method_name": method_name,
    "dataset_name": dataset_name,
    "split_type": split_type,
    "num_clients": num_clients,
    "num_federated_layers": num_federated_layers,
    "num_deep_unfolding_iterations": num_deep_unfolding_iterations,
    "initial_learning_rate": initial_learning_rate,
    "initial_perturbation": initial_perturbation,
    "fedprox_mu": fedprox_mu if method_name == "fedprox" else None,
    "scaffold_lr": scaffold_lr if method_name == "scaffold" else None,
    "local_maxiter": local_maxiter,
    "global_seed": global_seed,
    "results_dir": method_root,
    "global_csv_file": global_csv_file,
    "validation_csv_file": validation_csv_file,
    "round_metrics_csv_file": round_metrics_csv_file,
    "client_trace_csv_file": client_trace_csv_file,
    "subset_csv_file": subset_csv_file,
    "subset_centroid_csv_file": subset_centroid_csv_file,
    "outer_meta_csv_file": outer_meta_csv_file,
    "global_params_npz_file": global_params_npz_file,
}

with open(os.path.join(method_root, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)

def print_run_config(run_config, method_root):
    print("\n" + "="*70)
    print("CURRENT RUN CONFIGURATION")
    print("="*70)
    for k, v in run_config.items():
        print(f"{k}: {v}")
    print("="*70 + "\n")

print_run_config(run_config, method_root)


# ---------------- run ----------------
metrics = metrics_init(log_path=round_metrics_csv_file)

global_acc, clients_train, clients_test, round_times, val_losses, info_last = run_federated_training_meta(
    clients=clients,
    num_federated_layers=num_federated_layers,
    num_deep_unfolding_iterations=num_deep_unfolding_iterations,
    initial_learning_rate=initial_learning_rate,
    initial_perturbation=initial_perturbation,
    num_features=num_features,
    global_csv_file=global_csv_file,
    validation_csv_file=validation_csv_file,
    test_sequences=test_sequences,
    test_labels=test_labels,
    X_val=X_val,
    y_val=y_val,
    metrics=metrics,

    use_deep_unfolding=use_deep_unfolding,
    use_outer_meta=use_outer_meta,
    meta_cfg=meta_cfg,
    global_seed=global_seed,

    client_trace_csv_file=client_trace_csv_file,
    subset_csv_file=subset_csv_file,
    subset_centroid_csv_file=subset_centroid_csv_file,
    outer_meta_csv_file=outer_meta_csv_file,
    global_params_npz_file=global_params_npz_file,

    extra_config=extra_config,
)

/usr/local/lib/python3.12/dist-packages/genomic_benchmarks/utils/datasets.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Qiskit: 2.4.1
qiskit_aer available?: True


/tmp/ipykernel_6573/4165853264.py:55: DeprecationWarning: The class ``qiskit.circuit.library.n_local.real_amplitudes.RealAmplitudes`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.real_amplitudes instead.
  param_dim = len(RealAmplitudes(num_qubits=len(test_sequences[0]), reps=3).parameters)



CURRENT RUN CONFIGURATION
method_name: duqfl
dataset_name: Genome
split_type: NONIID
num_clients: 10
num_federated_layers: 10
num_deep_unfolding_iterations: 5
initial_learning_rate: 0.13
initial_perturbation: 0.13
fedprox_mu: None
scaffold_lr: None
local_maxiter: 25
global_seed: 42
results_dir: /content/drive/MyDrive/InitialDQFL_Project/Results/results/duqfl/Genome_NONIID_20260504_202737
global_csv_file: /content/drive/MyDrive/InitialDQFL_Project/Results/results/duqfl/Genome_NONIID_20260504_202737/global_accuracies.csv
validation_csv_file: /content/drive/MyDrive/InitialDQFL_Project/Results/results/duqfl/Genome_NONIID_20260504_202737/validation.csv
round_metrics_csv_file: /content/drive/MyDrive/InitialDQFL_Project/Results/results/duqfl/Genome_NONIID_20260504_202737/round_metrics.csv
client_trace_csv_file: /content/drive/MyDrive/InitialDQFL_Project/Results/results/duqfl/Genome_NONIID_20260504_202737/client_unfold_trace.csv
subset_csv_file: /content/drive/MyDrive/InitialDQFL_Project/Resu

Training Progress:   0%|          | 0/10 [00:00<?, ?it/s]

[DEBUG] round=0, unfold=0, param_shift=0.271249, loss_before=0.622649, loss_after=0.586360, loss_delta=0.036289, train_acc=0.7805, test_acc=0.5065, num_callbacks=0
[DEBUG] round=0, unfold=1, param_shift=0.154570, loss_before=0.586360, loss_after=0.574164, loss_delta=0.012196, train_acc=0.7805, test_acc=0.5060, num_callbacks=0
[DEBUG] round=0, unfold=2, param_shift=0.168785, loss_before=0.574164, loss_after=0.556921, loss_delta=0.017242, train_acc=0.7805, test_acc=0.5295, num_callbacks=0
[DEBUG] round=0, unfold=3, param_shift=0.246427, loss_before=0.556921, loss_after=0.532424, loss_delta=0.024498, train_acc=0.8049, test_acc=0.5055, num_callbacks=0
[DEBUG] round=0, unfold=4, param_shift=0.140362, loss_before=0.532424, loss_after=0.521624, loss_delta=0.010800, train_acc=0.8049, test_acc=0.5055, num_callbacks=0
  [Round 0 | Client 0] train_acc=0.8049, test_acc=0.5055, final_lr=0.128663, final_pert=0.131561, loss_last=0.521624, loss_delta=0.010800, accept=0.0000
[DEBUG] round=0, unfold=0, 

Training Progress:  10%|█         | 1/10 [1:39:43<14:57:27, 5983.09s/it]

[Round   0] acc_g=0.637 (μ=0.614, σ=0.081, FG=0.200) | t=5939.118s, val=0.638, meta=0.638 | subset=duqfl_fedavg_debug, eff_n=10.000, top1=0.100
[Round 0] global_acc=0.6375 val_loss=0.6376 meta_loss=0.6376
[DEBUG] round=1, unfold=0, param_shift=0.152721, loss_before=0.636757, loss_after=0.621605, loss_delta=0.015152, train_acc=0.7188, test_acc=0.6685, num_callbacks=0
[DEBUG] round=1, unfold=1, param_shift=0.169838, loss_before=0.621605, loss_after=0.604118, loss_delta=0.017486, train_acc=0.7500, test_acc=0.6715, num_callbacks=0
[DEBUG] round=1, unfold=2, param_shift=0.240847, loss_before=0.604118, loss_after=0.577605, loss_delta=0.026513, train_acc=0.8281, test_acc=0.6580, num_callbacks=0
[DEBUG] round=1, unfold=3, param_shift=0.170941, loss_before=0.577605, loss_after=0.556519, loss_delta=0.021086, train_acc=0.8281, test_acc=0.6445, num_callbacks=0
[DEBUG] round=1, unfold=4, param_shift=0.175033, loss_before=0.556519, loss_after=0.538064, loss_delta=0.018455, train_acc=0.8281, test_acc

Training Progress:  20%|██        | 2/10 [3:24:22<13:40:59, 6157.48s/it]

[Round   1] acc_g=0.683 (μ=0.643, σ=0.058, FG=0.149) | t=6233.563s, val=0.620, meta=0.620 | subset=duqfl_fedavg_debug, eff_n=10.000, top1=0.100
[Round 1] global_acc=0.6830 val_loss=0.6198 meta_loss=0.6198
[DEBUG] round=2, unfold=0, param_shift=0.138576, loss_before=0.634090, loss_after=0.619537, loss_delta=0.014552, train_acc=0.7000, test_acc=0.6420, num_callbacks=0
[DEBUG] round=2, unfold=1, param_shift=0.218594, loss_before=0.619537, loss_after=0.587372, loss_delta=0.032165, train_acc=0.8000, test_acc=0.6510, num_callbacks=0
[DEBUG] round=2, unfold=2, param_shift=0.209310, loss_before=0.587372, loss_after=0.564486, loss_delta=0.022886, train_acc=0.8333, test_acc=0.6565, num_callbacks=0
[DEBUG] round=2, unfold=3, param_shift=0.144590, loss_before=0.564486, loss_after=0.550161, loss_delta=0.014324, train_acc=0.8000, test_acc=0.6485, num_callbacks=0
[DEBUG] round=2, unfold=4, param_shift=0.162205, loss_before=0.550161, loss_after=0.535932, loss_delta=0.014229, train_acc=0.8000, test_acc